# WormML — LoopBio Camera Pipeline

This notebook trains a YOLOv11 worm-counting model on your own annotated images from the **LoopBio** imaging system.

**What you need before starting:**
- A Google account (to save data and weights in Google Drive)
- A CVAT export `.zip` file (export format: Ultralytics YOLO Detection 1.0, Save Images on)
- A GPU runtime — go to **Runtime → Change runtime type → T4 GPU** now

**No coding knowledge required. Just run each cell top to bottom.**

## Step 0 — Check GPU

Before anything else, make sure you have a GPU assigned.
Go to **Runtime → Change runtime type → T4 GPU**.

Then run this cell to confirm:

In [ ]:
import torch
if torch.cuda.is_available():
    print(f"✅ GPU ready: {torch.cuda.get_device_name(0)}")
else:
    print("❌ No GPU detected. Go to Runtime → Change runtime type → T4 GPU")
    raise SystemExit("Please enable GPU before continuing.")

## Step 1 — Install WormML

This downloads the WormML code and installs all required packages.
It only needs to run once per Colab session (re-run if you restart the runtime).

In [ ]:
!git clone https://github.com/tommyli88/wormml.git
%cd wormml
!pip install -q -r requirements.txt
print("✅ Installation complete")

## Step 2 — Connect Google Drive

Your data and trained weights will be saved to Google Drive so they survive when the Colab session ends.

Run the cell — a link will appear. Click it and follow the prompts to authorise.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

import os

# Where to store everything in your Drive — change this if you like
BASE = '/content/drive/MyDrive/wormml_lb'

for folder in ['raw/images', 'raw/labels', 'split', 'preprocessed', 'runs']:
    os.makedirs(f'{BASE}/{folder}', exist_ok=True)

print(f"✅ Working folder: {BASE}")

## Step 3 — Upload Your CVAT Export

In CVAT:
1. Open your task
2. Click **Export dataset**
3. Format: **Ultralytics YOLO Detection 1.0**
4. Tick **Save Images**
5. Download the `.zip` file

Then run the cell below and click **Choose Files** to upload that zip.

In [ ]:
from google.colab import files
import zipfile, shutil, pathlib

uploaded = files.upload()   # click 'Choose Files' and select your CVAT zip
zip_name = list(uploaded.keys())[0]
print(f"Uploaded: {zip_name}")

# Unzip into a temporary folder
extract_dir = '/content/cvat_export'
if os.path.exists(extract_dir):
    shutil.rmtree(extract_dir)
with zipfile.ZipFile(zip_name, 'r') as z:
    z.extractall(extract_dir)
print("✅ Unzipped successfully")

## Step 4 — Prepare the Data

CVAT puts all images inside `images/train/` and all labels inside `labels/train/`. This cell moves them up one level into the flat layout WormML expects, then saves them to your Google Drive.

In [ ]:
src_imgs = pathlib.Path(extract_dir) / 'images' / 'train'
src_lbls = pathlib.Path(extract_dir) / 'labels' / 'train'
dst_imgs = pathlib.Path(f'{BASE}/raw/images')
dst_lbls = pathlib.Path(f'{BASE}/raw/labels')

# Clear destination folders
for d in [dst_imgs, dst_lbls]:
    if d.exists(): shutil.rmtree(d)
    d.mkdir(parents=True)

# Copy
img_count = 0
for f in src_imgs.iterdir():
    shutil.copy2(f, dst_imgs / f.name)
    img_count += 1

lbl_count = 0
for f in src_lbls.glob('*.txt'):
    shutil.copy2(f, dst_lbls / f.name)
    lbl_count += 1

print(f"✅ {img_count} images → {dst_imgs}")
print(f"✅ {lbl_count} labels → {dst_lbls}")
if img_count != lbl_count:
    print(f"⚠️  {img_count - lbl_count} images have no matching label and will be skipped.")

## Step 5 — Split into Train / Val

Splits your data 80% training / 20% validation using seed 42. This is reproducible — running it again gives the exact same split.

In [ ]:
import sys
sys.path.insert(0, '/content/wormml')

from scripts.split_dataset import split_dataset

split_dataset(
    input_dir   = f'{BASE}/raw',
    output_dir  = f'{BASE}/split',
    train_ratio = 0.8,
    seed        = 42,
)

## Step 6 — Preprocess

Detects and crops the petri dish, then **inverts the colours** (LoopBio images are dark-on-bright; YOLO works best bright-on-dark). On the training split it also saves an augmented copy of each image with random brightness, blur, and noise — effectively doubling your training set.

In [ ]:
from wormml.preprocess import preprocess_dataset, get_config

cfg = get_config("lb")
preprocess_dataset(
    input_base_dir  = f'{BASE}/split',
    output_base_dir = f'{BASE}/preprocessed',
    cfg             = cfg,
    visualize       = True,
)

## Step 7 — Train

Trains YOLOv11 Large for 100 epochs using the hyperparameters tuned for the LoopBio camera.

⏱ **Expect 2–4 hours on a Colab T4 GPU.** The cell will print progress every epoch. Your weights are saved to Google Drive automatically when training finishes.

In [ ]:
from wormml.train import TrainingConfig, train

train_cfg = TrainingConfig(
    camera       = "lb",
    dataset_base = f'{BASE}/preprocessed',
    output_dir   = f'{BASE}/runs',
)
train(train_cfg)

weights_path = f'{BASE}/runs/yolov11_maxacc_11l/weights/best.pt'
print(f"\n✅ Best weights saved to: {weights_path}")

## Step 8 — Find Optimal Thresholds

Searches for the confidence and IoU threshold combination that gives the lowest counting error on your validation set. Always run this after training — don't use default values.

In [ ]:
from wormml.threshold import sweep_thresholds

best = sweep_thresholds(
    model_path = f'{BASE}/runs/yolov11_maxacc_11l/weights/best.pt',
    images_dir = f'{BASE}/preprocessed/images/val',
    labels_dir = f'{BASE}/preprocessed/labels/val',
)
print(f"Best confidence: {best['conf']:.3f}")
print(f"Best IoU:        {best['iou']:.3f}")
print(f"MAE:             {best['mae']:.2f}")

## Step 9 — Evaluate

Runs the full evaluation on your validation set using Hungarian matching to pair predictions to ground-truth annotations. Reports MAE, RMSE, precision, recall, and F1.

In [ ]:
from wormml.evaluate import EvalConfig, evaluate, print_summary_tables

results = evaluate([EvalConfig(
    camera     = "LB",
    model_path = f'{BASE}/runs/yolov11_maxacc_11l/weights/best.pt',
    images_dir = f'{BASE}/preprocessed/images/val',
    labels_dir = f'{BASE}/preprocessed/labels/val',
)])
print_summary_tables(results)

## ✅ Done

Your trained weights are at:
```
MyDrive/wormml_lb/runs/yolov11_maxacc_11l/weights/best.pt
```

Download them from the **Files** panel on the left (click the folder icon) or find them in Google Drive.

To run inference on new images, load the weights with:
```python
from ultralytics import YOLO
model = YOLO('wormml_lb_best.pt')
results = model('your_image.jpg', conf=0.35)
print(f'Worm count: {len(results[0].boxes)}')
```